# TIGeR Kaggle Workflow

This notebook executes the complete Text-Image Generative Repair (TIGeR) pipeline.

### ⚠️ Setup Instructions Before You Start:
1. **Turn on the GPU:** Go to the right sidebar -> `Notebook options` -> `Accelerator` -> select **GPU T4 x2** or **P100**.
2. **Add your Gemini API Key:** Go to the right sidebar -> `Add-ons` -> `Secrets`. Add a new secret named `GEMINI_API_KEY` and paste your key.

In [ ]:
!rm -rf TIGeR-Text-Image-Generative-Repair
!git clone https://github.com/namaray/TIGeR-Text-Image-Generative-Repair.git
%cd TIGeR-Text-Image-Generative-Repair
!pip install -e ".[dev,vlm]" -q

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

try:
    user_secrets = UserSecretsClient()
    gemini_key = user_secrets.get_secret("GEMINI_API_KEY")
    os.environ["GEMINI_API_KEY"] = gemini_key
    print("✅ Gemini API Key loaded successfully from Kaggle Secrets!")
except Exception as e:
    print("❌ Failed to load Gemini API Key. Did you add it to Kaggle Secrets? Error:", e)

## 1. Data Generation (Synthgen)

In [ ]:
!python -m tiger.cli synthgen

## 2. Calibration & Training

In [ ]:
!python -m tiger.cli calibrate

In [ ]:
!python -m tiger.cli train-arbiter

In [ ]:
!python -m tiger.cli calibrate-fusion

## 3. The Pipeline (Detect -> Route -> Repair)

In [ ]:
!python -m tiger.cli noise --seed 7

In [ ]:
!python -m tiger.cli detect --seed 7

In [ ]:
!python -m tiger.cli analyze --seed 7

In [ ]:
!python -m tiger.cli route --seed 7

### Final Step: Closed-Loop Repair (with Gemini VLM Judge)

In [ ]:
!python -m tiger.cli repair --seed 7 --vlm-judge

## 4. Export Outputs

In [ ]:
!zip -r /kaggle/working/tiger_outputs.zip data/outputs data/thresholds data/processed
print("✅ Download tiger_outputs.zip from the '/kaggle/working' directory in the right sidebar!")